<a href="https://colab.research.google.com/github/soleildayana/Apophis-Asteroid-Project/blob/main/dart_inverso/nb00_parches_conicos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NB00 — Misión DART Inverso: Jerarquía de Parches Cónicos
## Asteroide (99942) Apophis — Intercepción y Redirección hacia la Tierra

**Autor:** Soleil Dayana Niño Murcia — 1033097666  
**Curso:** Mecánica Celeste  
**Fecha:** Mayo 2026

---

> **Objetivo:** Modelar una misión hipotética de tipo DART Inverso que impacte a Apophis para desviar
> su trayectoria y dirigirla hacia la Tierra. Usando el método de **parches cónicos** (*patched conics*),
> se descompone la misión en tres fases independientes: escape geocéntrico del impactor, intercepción
> heliocéntrica de Apophis, y el encuentro hiperbólico final con la Tierra. Este modelo provee las
> magnitudes de $\Delta V$ necesarias y los parámetros orbitales de cada fase.

## 1. Marco Teórico: El Método de Parches Cónicos

### 1.1 Fundamento

El método de **parches cónicos** (*patched-conic approximation*) simplifica trayectorias
interplanetarias dividiendo el espacio en **esferas de influencia gravitacional** (SOI), dentro
de cada cual solo domina la atracción de un cuerpo. Las órbitas en cada zona son **secciones
cónicas puras** (elipses, parábolas, hipérbolas), sin perturbaciones.

En cada **parche** se resuelve el problema de los **dos cuerpos** de forma analítica:
$$\ddot{\mathbf{r}} = -\frac{\mu}{r^3}\,\mathbf{r}$$

Las condiciones en los **límites** de cada zona (esfera de Hill) son el pegamento que une
las tres fases.

### 1.2 Las tres fases de la misión DART Inverso

| Fase | Sistema gravitacional | Cuerpo central | Tipo de cónica | Condición de éxito |
|------|-----------------------|---------------|-----------------|--------------------|
| **0** — Escape geocéntrico | Tierra–impactor | Tierra ($M_\oplus$) | Hipérbola / parábola de escape | $\varepsilon \geq 0$ (energía ≥ 0) |
| **1** — Intercepción heliocéntrica | Sol–Apophis | Sol ($M_\odot$) | Elipse/transferencia | $\Delta V$ aplicado en Apophis; órbita intersecta la Tierra |
| **2** — Encuentro hiperbólico | Tierra–Apophis | Tierra ($M_\oplus$) | Hipérbola | Periapsis $q < R_\oplus$ |

### 1.3 Esfera de Hill terrestre

La **esfera de influencia de Hill** delimita la región donde la atracción terrestre
supera a la perturbación solar:

$$\boxed{R_H = a_\oplus \left(\frac{M_\oplus}{3\,M_\odot}\right)^{1/3}}$$

Con $a_\oplus \approx 1\,\text{AU}$, $M_\oplus/M_\odot \approx 3\times10^{-6}$:
$$R_H \approx 0.0100\,\text{AU} \approx 1.5 \times 10^6\,\text{km} \approx 3.9\,d_{\text{Tierra-Luna}}$$

### 1.4 Ecuación de Vis-Viva

En cualquier punto de una cónica kepleriana, la velocidad cumple la **ecuación de Vis-Viva**:

$$\boxed{v^2 = \mu\left(\frac{2}{r} - \frac{1}{a}\right)}$$

donde $a$ es el semieje mayor de la órbita ($a < 0$ para hipérbolas). De aquí se
derivan directamente las velocidades de escape ($a \to \infty$) y circular:

$$v_{esc} = \sqrt{\frac{2\mu}{r}}, \qquad v_{circ} = \sqrt{\frac{\mu}{r}}$$

### 1.5 Problema de Lambert (Fase 1)

Dado que conocemos la posición de Apophis en el momento del impacto DART ($\mathbf{r}_1$)
y la posición de la Tierra en el momento del encuentro ($\mathbf{r}_2$), el **problema de Lambert**
determina las velocidades inicial y final de una cónica que une ambos puntos en un tiempo $\tau$:

$$\text{Dado: } \mathbf{r}_1,\; \mathbf{r}_2,\; \tau \quad\Rightarrow\quad \mathbf{V}_1,\; \mathbf{V}_2$$

El **impulso** necesario en Apophis es:
$$\Delta V = \|\mathbf{V}_1 - \mathbf{v}_{\text{Apophis}}\|$$

### 1.6 Encuentro hiperbólico geocéntrico (Fase 2)

Cuando Apophis entra en la esfera de Hill terrestre con velocidad hiperbólica $v_\infty$,
su trayectoria es una **hipérbola geocéntrica** con:

$$a_h = -\frac{\mu_\oplus}{v_\infty^2} \quad (\text{negativo})$$

$$e = \sqrt{1 + \frac{b^2\,v_\infty^4}{\mu_\oplus^2}}$$

$$q = |a_h|(e - 1)$$

donde $b$ es el **parámetro de impacto**. El impacto ocurre si $q < R_\oplus$.

In [ ]:
%pip install -Uq pymcel

## 2. Configuración: Unidades Canónicas y Constantes

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import pymcel as pc
from pymcel import constantes as const

# ── Unidades canónicas (Sol, AU, UT) ──────────────────────────────────────
UL = const.au          # 1 AU en metros
UM = const.M_sun       # masa del Sol en kg
G  = const.G           # constante gravitacional SI
UT = np.sqrt(UL**3 / (G * UM))  # tiempo canónico ≈ 5.022e6 s ≈ 58.13 días
UT_days = UT / 86400   # ≈ 58.13 días

mu_sol    = 1.0                              # parámetro gravitacional del Sol (canónico)
mu_tierra = const.GM_earth / (UL**3/UT**2)  # parámetro gravitacional de la Tierra (canónico)

# Radio y esfera de Hill de la Tierra (en AU)
R_tierra_km = 6371.0
R_tierra_AU = R_tierra_km * 1e3 / UL

# Esfera de Hill terrestre
a_tierra = 1.0   # ~1 AU
R_H = a_tierra * (const.M_earth / (3 * const.M_sun))**(1/3)
R_H_km = R_H * UL / 1e3

print(f"UT = {UT_days:.4f} días")
print(f"mu_tierra (canónico) = {mu_tierra:.4e}")
print(f"R_tierra = {R_tierra_AU:.4e} AU = {R_tierra_km:.1f} km")
print(f"Esfera de Hill terrestre R_H = {R_H:.4e} AU = {R_H_km:.0f} km")

## 3. Fase 0 — Escape Geocéntrico del Impactor

### 3.1 Teoría

Antes de poder interceptar a Apophis en el espacio heliocéntrico, el impactor debe **escapar
del pozo gravitacional terrestre**. Partiendo desde una Órbita Baja Terrestre (LEO), necesita
alcanzar al menos la velocidad de escape.

La **energía específica** de una órbita geocéntrica es:
$$\varepsilon = \frac{v^2}{2} - \frac{\mu_\oplus}{r}$$

- $\varepsilon < 0$: órbita **elíptica** (ligada)
- $\varepsilon = 0$: trayectoria **parabólica** (límite de escape)
- $\varepsilon > 0$: trayectoria **hiperbólica** (escapada con exceso de velocidad $v_\infty = \sqrt{2\varepsilon}$)

El **$\Delta V$ mínimo** para escapar desde LEO es:
$$\Delta V_{esc} = v_{esc} - v_{circ} = \sqrt{\frac{2\mu_\oplus}{r_{LEO}}} - \sqrt{\frac{\mu_\oplus}{r_{LEO}}} = v_{circ}\,(\sqrt{2}-1)$$

Este $\Delta V$ representa el **límite inferior** de la maniobra de escape; en la práctica
se necesita un poco más para enviar el impactor hacia Apophis con la velocidad correcta.

In [ ]:
# ── Fase 0: Escape geocéntrico del impactor ───────────────────────────────
# Condiciones iniciales del impactor en órbita baja terrestre (LEO)
h_LEO_km   = 400.0                         # altitud LEO típica [km]
r_LEO_km   = R_tierra_km + h_LEO_km        # radio desde centro Tierra [km]
r_LEO_m    = r_LEO_km * 1e3

# Velocidades en m/s
v_circ_LEO = np.sqrt(const.GM_earth / r_LEO_m)          # circular
v_esc_LEO  = np.sqrt(2 * const.GM_earth / r_LEO_m)      # escape

# Δv mínimo para escapar desde LEO
dv_escape  = v_esc_LEO - v_circ_LEO

print("─── Fase 0: Escape Geocéntrico ───────────────────────────────")
print(f"Radio LEO        = {r_LEO_km:.1f} km")
print(f"v_circular LEO   = {v_circ_LEO/1e3:.4f} km/s")
print(f"v_escape LEO     = {v_esc_LEO/1e3:.4f} km/s")
print(f"ΔV mínimo escape = {dv_escape/1e3:.4f} km/s")
print(f"Esfera de Hill   = {R_H_km:.0f} km ≈ {R_H_km/384400:.2f} distancias Tierra-Luna")

# Graficar v_esc vs altitud
alt_km = np.linspace(200, 2000, 200)
r_m    = (R_tierra_km + alt_km) * 1e3
v_esc  = np.sqrt(2 * const.GM_earth / r_m) / 1e3
v_cir  = np.sqrt(const.GM_earth / r_m) / 1e3
dv_arr = v_esc - v_cir

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(alt_km, v_esc, 'b-', label='$v_{esc}$')
ax.plot(alt_km, v_cir, 'g--', label='$v_{circ}$')
ax.plot(alt_km, dv_arr, 'r:', label='$\\Delta v_{escape}$')
ax.axvline(h_LEO_km, color='k', ls=':', lw=1, label=f'LEO ({h_LEO_km} km)')
ax.set_xlabel('Altitud [km]')
ax.set_ylabel('Velocidad [km/s]')
ax.set_title('Fase 0 — Velocidades geocéntricas vs altitud')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Fase 1 — Intercepción Heliocéntrica de Apophis

### 4.1 Estrategia de intercepción

Una vez que el impactor escapa de la Tierra, entra al dominio heliocéntrico. El objetivo es
que el impactor **modifique la trayectoria de Apophis** en un instante $t_{DART}$, aplicando
un impulso $\Delta V$ que redirija a Apophis hacia la Tierra.

El problema se plantea de forma inversa:
- **Conocemos** la posición de Apophis en $t_{DART}$ y la posición de la Tierra en $t_{enc}$
- **Queremos** la velocidad que debe tener Apophis justo después del impacto DART para que
  llegue a la Tierra en $\tau = t_{enc} - t_{DART}$ días

Esto es exactamente el **Problema de Lambert**:

$$\text{Dado: } \mathbf{r}_{Apophis}(t_{DART}),\; \mathbf{r}_{\oplus}(t_{enc}),\; \tau
\quad\Rightarrow\quad \mathbf{V}_1,\; \mathbf{V}_2$$

El impulso necesario en Apophis vale:
$$\Delta V = \|\mathbf{V}_1 - \mathbf{v}_{Apophis}(t_{DART})\|$$

### 4.2 El encuentro de 2029

El **13 de abril de 2029**, Apophis tendrá un encuentro muy cercano con la Tierra
($\approx 38{,}000$ km), lo que lo convierte en el candidato ideal para este análisis:
pequeñas perturbaciones en su trayectoria tienen efectos amplificados cerca del perigeo.

### 4.3 Análisis de ventana temporal

El parámetro $\tau$ (tiempo de antelación del impacto DART) es clave:
- **$\tau$ grande** (meses antes): $\Delta V$ pequeño, pero menor precisión de efemérides
- **$\tau$ pequeño** (días antes): $\Delta V$ enorme (intercepción casi perpendicular)

Barremos $\tau \in \{30, 60, 90, 120, 180, 270, 365\}$ días para caracterizar la ventana.

In [ ]:
# ── Fase 1: Efemérides de Apophis y la Tierra ─────────────────────────────
# Fecha del encuentro cercano de 2029
t_encuentro = '2029-04-13'

# Rango de fechas para Apophis (6 meses antes del encuentro)
epochs_apophis = {
    'start': '2028-10-13',
    'stop':  '2029-04-13',
    'step':  '1d'
}

print("Consultando efemérides de Apophis (99942)...")
df_apophis = pc.consulta_horizons(
    id='99942', location='@0',
    epochs=epochs_apophis, datos='vectors'
)
print(f"Apophis: {len(df_apophis)} épocas, columnas: {list(df_apophis.columns[:8])}")

print("\nConsultando efemérides de la Tierra (399)...")
df_tierra = pc.consulta_horizons(
    id='399', location='@0',
    epochs=epochs_apophis, datos='vectors'
)
print(f"Tierra: {len(df_tierra)} épocas")

# Convertir velocidades de AU/día a AU/UT (unidades canónicas)
# 1 AU/día × UT_days = AU/UT
for df in [df_apophis, df_tierra]:
    df['vx_can'] = df['vx'] * UT_days
    df['vy_can'] = df['vy'] * UT_days
    df['vz_can'] = df['vz'] * UT_days

print("\nConversión a unidades canónicas aplicada (AU/UT = AU/día × UT_days)")

In [ ]:
# ── Problema de Lambert inverso (tau = 180 días) ──────────────────────────
tau_dias = 180    # tiempo de antelación del impacto en días
tau_UT   = tau_dias / UT_days   # en unidades canónicas de tiempo

# Estado de Apophis en t_dart = t_encuentro - tau
idx_dart  = -tau_dias - 1   # 180 días antes del último registro (día 0 = 2029-04-13)
fila_dart = df_apophis.iloc[idx_dart]

r_A = np.array([fila_dart['x'],  fila_dart['y'],  fila_dart['z']])
v_A = np.array([fila_dart['vx_can'], fila_dart['vy_can'], fila_dart['vz_can']])

# Estado de la Tierra en t_encuentro
fila_enc = df_tierra.iloc[-1]
r_E = np.array([fila_enc['x'],  fila_enc['y'],  fila_enc['z']])
v_E = np.array([fila_enc['vx_can'], fila_enc['vy_can'], fila_enc['vz_can']])

print(f"τ = {tau_dias} días | t_DART = {fila_dart['datetime_str']}")
print(f"r_Apophis DART: {r_A} AU")
print(f"r_Tierra ENC:   {r_E} AU")
print(f"Separación:     {np.linalg.norm(r_A - r_E):.4f} AU")

# Resolver problema de Lambert
V1_new, V2_new, info_lambert = pc.solucion_lambert(
    r_A, r_E, tau_UT, mu=mu_sol, direccion='pro'
)

# ΔV necesario en Apophis
dv_vec  = V1_new - v_A
dv_mag  = np.linalg.norm(dv_vec)

# Semieje mayor de la nueva órbita (vis-viva en dos puntos)
r_A_mag    = np.linalg.norm(r_A)
r_E_mag    = np.linalg.norm(r_E)
v1_new_mag = np.linalg.norm(V1_new)
eps_new    = v1_new_mag**2 / 2 - mu_sol / r_A_mag
a_new      = -mu_sol / (2 * eps_new)

print(f"\n─── Lambert (τ = {tau_dias} d) ─────────────────────────────────────")
print(f"V_Apophis original:  {np.linalg.norm(v_A):.6f} AU/UT")
print(f"V_Apophis nueva:     {v1_new_mag:.6f} AU/UT")
print(f"ΔV (canónico):       {dv_mag:.6f} AU/UT")
print(f"ΔV (km/s):           {dv_mag * UL / UT / 1e3:.4f} km/s")
print(f"Semieje mayor nuevo: {a_new:.4f} AU")
print(f"Vis-viva check  ε:   {eps_new:.6f} AU²/UT²")

## 5. Ventana de Tiempo — $\Delta V(\tau)$

### 5.1 Razonamiento físico

La magnitud de $\Delta V$ depende fuertemente de la geometría orbital en el momento del
impacto DART. Intuitivamente:

- Si $\tau$ es **grande** (impacto DART meses antes del encuentro), la órbita de transferencia
  tiene parámetros similares a la órbita natural de Apophis → $\Delta V$ pequeño

- Si $\tau$ es **pequeño** (impacto días antes), se necesita una trayectoria casi perpendicular
  a la órbita de Apophis → $\Delta V$ enorme

La **velocidad hiperbólica de llegada** $v_\infty$ (velocidad relativa de Apophis respecto
a la Tierra al llegar) también varía con $\tau$ y determina la viabilidad de la Fase 2:

$$v_\infty = \|\mathbf{V}_2 - \mathbf{v}_\oplus(t_{enc})\|$$

Un $v_\infty$ alto facilita que el periapsis $q$ sea pequeño (mayor probabilidad de impacto),
pero un $v_\infty$ bajo implica mayor amplificación gravitacional al caer hacia la Tierra.

In [ ]:
# ── Ventana de tiempo: ΔV en función de τ ────────────────────────────────
tau_valores_dias = [30, 60, 90, 120, 180, 270, 365]
resultados = []

for tau_d in tau_valores_dias:
    tau_ut = tau_d / UT_days
    idx    = max(-tau_d - 1, -len(df_apophis))

    fila = df_apophis.iloc[idx]
    r_Ai = np.array([fila['x'], fila['y'], fila['z']])
    v_Ai = np.array([fila['vx_can'], fila['vy_can'], fila['vz_can']])

    try:
        V1i, V2i, _ = pc.solucion_lambert(r_Ai, r_E, tau_ut, mu=mu_sol, direccion='pro')
        dv_i = np.linalg.norm(V1i - v_Ai)
        dv_i_kms = dv_i * UL / UT / 1e3

        # Velocidad hiperbólica en llegada (relativa a la Tierra)
        v_inf_i   = np.linalg.norm(V2i - v_E)
        v_inf_kms = v_inf_i * UL / UT / 1e3

        resultados.append({
            'tau_dias':   tau_d,
            'dv_AU_UT':   dv_i,
            'dv_km_s':    dv_i_kms,
            'v_inf_km_s': v_inf_kms,
            'fecha_dart': fila['datetime_str']
        })
        print(f"τ = {tau_d:4d} d | ΔV = {dv_i_kms:.4f} km/s | v∞ = {v_inf_kms:.3f} km/s | fecha: {fila['datetime_str']}")
    except Exception as ex:
        print(f"τ = {tau_d:4d} d | Error Lambert: {ex}")

df_res = pd.DataFrame(resultados)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Panel izquierdo: ΔV vs τ
ax = axes[0]
ax.semilogy(df_res['tau_dias'], df_res['dv_km_s'], 'bo-', lw=2, ms=8, label='ΔV calculado')
ax.set_xlabel('Tiempo de antelación τ [días]', fontsize=12)
ax.set_ylabel('ΔV [km/s]', fontsize=12)
ax.set_title('Ventana de tiempo: ΔV necesario en Apophis', fontsize=13)
ax.grid(True, which='both', alpha=0.3)
ax.legend(fontsize=11)

# Panel derecho: v_inf en llegada
ax2 = axes[1]
ax2.plot(df_res['tau_dias'], df_res['v_inf_km_s'], 'rs-', lw=2, ms=8, label='$v_\\infty$ geocéntrica')
ax2.set_xlabel('Tiempo de antelación τ [días]', fontsize=12)
ax2.set_ylabel('$v_\\infty$ relativa a Tierra [km/s]', fontsize=12)
ax2.set_title('Velocidad hiperbólica en llegada', fontsize=13)
ax2.grid(True, alpha=0.3)
ax2.legend(fontsize=11)

plt.suptitle('Misión DART Inverso — Análisis de Ventana Temporal', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print("\nResumen:")
print(df_res[['tau_dias','dv_km_s','v_inf_km_s','fecha_dart']].to_string(index=False))

## 6. Fase 2 — Encuentro Hiperbólico Tierra-Apophis

### 6.1 Geometría de la hipérbola geocéntrica

Una vez que Apophis cruza la esfera de Hill terrestre con velocidad relativa $v_\infty$,
su trayectoria en el sistema de referencia geocéntrico es una **hipérbola**.

Los parámetros fundamentales de esta hipérbola son:

| Parámetro | Expresión | Descripción |
|-----------|-----------|-------------|
| $a_h$ | $-\mu_\oplus / v_\infty^2$ | Semieje mayor (negativo para hipérbola) |
| $h$ | $b \cdot v_\infty$ | Momento angular específico |
| $e$ | $\sqrt{1 + (h\,v_\infty/\mu_\oplus)^2}$ | Excentricidad (>1 para hipérbola) |
| $q$ | $|a_h|(e-1)$ | Periapsis (distancia mínima al centro de la Tierra) |

### 6.2 Condición de impacto

El impacto ocurre si y solo si el periapsis es menor que el radio terrestre:
$$q < R_\oplus \quad\Longleftrightarrow\quad b < b_{max}$$

donde el **parámetro de impacto máximo** $b_{max}$ puede obtenerse igualando $q = R_\oplus$:
$$b_{max} = R_\oplus\,\sqrt{1 + \frac{2\mu_\oplus}{R_\oplus\,v_\infty^2}}$$

Esta fórmula incorpora el efecto de **focalización gravitacional**: para $v_\infty$ bajo,
$b_{max}$ puede ser mucho mayor que $R_\oplus$ (la Tierra «captura» un área efectiva mayor).

### 6.3 Velocidad al cruzar la esfera de Hill

Al entrar a la esfera de Hill, la velocidad de Apophis aumenta por la atracción terrestre:
$$v_{R_H} = \sqrt{v_\infty^2 + \frac{2\mu_\oplus}{R_H}}$$

Para el peor caso (alineación perfecta, $b=0$), Apophis impacta con velocidad:
$$v_{impacto} = \sqrt{v_\infty^2 + \frac{2\mu_\oplus}{R_\oplus}}$$

In [ ]:
# ── Fase 2: Encuentro hiperbólico Tierra–Apophis ──────────────────────────
# Tomamos el tau con mayor antelación (menor ΔV) como caso de estudio
mejor = df_res.sort_values('tau_dias').iloc[-1]
tau_d_best    = mejor['tau_dias']
v_inf_best_kms = mejor['v_inf_km_s']

print(f"Caso de estudio: τ = {tau_d_best} días")
print(f"v_∞ relativa a Tierra = {v_inf_best_kms:.3f} km/s")

# Convertir v_inf a m/s para cálculos geocéntricos
v_inf_ms = v_inf_best_kms * 1e3

# Parámetros de la hipérbola geocéntrica
mu_E  = const.GM_earth    # m³/s²
R_H_m = R_H * UL          # radio esfera de Hill en metros

# Velocidad al cruzar la esfera de Hill
v_at_RH = np.sqrt(v_inf_ms**2 + 2*mu_E / R_H_m)

# Semieje mayor de la hipérbola geocéntrica
a_h = -mu_E / v_inf_ms**2    # negativo

print(f"\nVelocidad al cruzar R_H  = {v_at_RH/1e3:.3f} km/s")
print(f"a_h (semieje hipérbola)   = {a_h/1e3:.0f} km")

# Barrer el parámetro de impacto b [km]
b_km = np.linspace(0, 50000, 500)    # 0 a 50,000 km
b_m  = b_km * 1e3

# Momento angular geocéntrico: h = b * v_inf
h_geo = b_m * v_inf_ms               # m²/s

# Excentricidad
e_geo = np.sqrt(1 + (h_geo * v_inf_ms / mu_E)**2)

# Periapsis
q_m  = np.abs(a_h) * (e_geo - 1)    # metros
q_km = q_m / 1e3

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(b_km, q_km, 'b-', lw=2)
ax.axhline(R_tierra_km, color='green', ls='--', lw=1.5, label=f'$R_\\oplus$ = {R_tierra_km:.0f} km')
ax.axhline(0, color='gray', ls=':', lw=1)
ax.fill_between(b_km, 0, R_tierra_km, alpha=0.12, color='red', label='Zona de impacto ($q < R_\\oplus$)')

# Parámetro de impacto máximo
b_max_idx = np.where(q_km <= R_tierra_km)[0]
if len(b_max_idx) > 0:
    b_max = b_km[b_max_idx[-1]]
    ax.axvline(b_max, color='red', ls='--', lw=1.5, label=f'$b_{{max}}$ = {b_max:.0f} km')
    print(f"\nParámetro de impacto máximo: b_max ≈ {b_max:.0f} km")
    # Velocidad de impacto para b=0
    v_impact_ms = np.sqrt(v_inf_ms**2 + 2*mu_E / (R_tierra_km*1e3))
    print(f"Velocidad de impacto (b=0): {v_impact_ms/1e3:.3f} km/s")
else:
    print("No se alcanza impacto con este v_inf")

ax.set_xlabel('Parámetro de impacto $b$ [km]', fontsize=12)
ax.set_ylabel('Periapsis $q$ [km]', fontsize=12)
ax.set_title(f'Fase 2 — Hipérbola geocéntrica ($v_\\infty$ = {v_inf_best_kms:.2f} km/s)', fontsize=13)
ax.set_ylim(0, min(60000, q_km.max()))
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Diagrama de la Misión

### 7.1 Descripción del diagrama de parches

El diagrama siguiente resume visualmente la jerarquía de parches cónicos:

```
┌─────────────────────────────────────────────────────────────────┐
│                   ESPACIO HELIOCÉNTRICO                          │
│                                                                   │
│   ☀ Sol         ╔══════════════╗           ╔══════════════╗      │
│                 ║  FASE 0      ║           ║   FASE 2     ║      │
│                 ║ Escape LEO   ║           ║  Encuentro   ║      │
│                 ║  geocentric  ║           ║  hiperbólico ║      │
│   🌍 Tierra ─── ╚══════════════╝    →    ──╚══════════════╝      │
│                      ║                              ↑            │
│                      ║    FASE 1: Lambert           │            │
│                      ║  Intercepción                │            │
│                      ╚══════════════════════> ✦ Apophis          │
└─────────────────────────────────────────────────────────────────┘
```

El pegamento entre fases son las condiciones en la **frontera de la esfera de Hill**:
- **Fase 0 → 1**: El impactor sale de la esfera de Hill con $v_\infty^{(0)}$ y se transforma
  en un objeto heliocéntrico.
- **Fase 1 → 2**: Apophis, ya en la nueva trayectoria post-DART, entra en la esfera de Hill
  terrestre con $v_\infty^{(2)} = \|\mathbf{V}_2 - \mathbf{v}_\oplus\|$.

In [ ]:
# ── Diagrama esquemático de la misión ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 7))
ax.set_xlim(-2.2, 2.2)
ax.set_ylim(-2.2, 2.2)
ax.set_aspect('equal')
ax.set_facecolor('#0d1117')
fig.patch.set_facecolor('#0d1117')

# ── Sol ──
sol_circ = plt.Circle((0, 0), 0.10, color='#FFD700', zorder=5)
ax.add_patch(sol_circ)
ax.text(0, 0.15, '☀ Sol', color='#FFD700', ha='center', fontsize=10, fontweight='bold')

# ── Tierra y su esfera de Hill ──
r_tierra_plot = np.array([1.0, 0.0])
tierra_circ   = plt.Circle(r_tierra_plot, 0.04, color='#4488FF', zorder=5)
hill_circ     = plt.Circle(r_tierra_plot, 0.20, color='#4488FF', fill=False,
                            ls='--', lw=1.2, alpha=0.6, zorder=4)
ax.add_patch(tierra_circ)
ax.add_patch(hill_circ)
ax.text(r_tierra_plot[0], r_tierra_plot[1]+0.07, '🌍 Tierra', color='#4488FF',
        ha='center', fontsize=9, fontweight='bold')
ax.text(r_tierra_plot[0]+0.22, r_tierra_plot[1]-0.10, '$R_H$', color='#4488FF',
        fontsize=8, alpha=0.8)

# ── Órbita terrestre ──
theta_e = np.linspace(0, 2*np.pi, 300)
ax.plot(np.cos(theta_e), np.sin(theta_e), color='#4488FF', lw=0.8, alpha=0.4, zorder=2)

# ── Apophis: posición aproximada en t_DART (180 días antes) ──
if len(df_apophis) > 0:
    row_dart = df_apophis.iloc[-181]
    r_ap = np.array([row_dart['x'], row_dart['y']])
    ax.plot(r_ap[0], r_ap[1], '*', color='#FF6600', ms=14, zorder=6)
    ax.text(r_ap[0]+0.08, r_ap[1]+0.07, 'Apophis\n(t_DART)', color='#FF6600',
            fontsize=8, fontweight='bold')

    # Posición de Apophis en encuentro
    row_enc = df_apophis.iloc[-1]
    r_ap_enc = np.array([row_enc['x'], row_enc['y']])
    ax.plot(r_ap_enc[0], r_ap_enc[1], 'o', color='#FF2200', ms=10, zorder=6)
    ax.text(r_ap_enc[0]+0.06, r_ap_enc[1]-0.12, 'Apophis\n(encuentro)', color='#FF2200',
            fontsize=8, fontweight='bold')

    # Órbita natural de Apophis (parte visible)
    xs_ap = df_apophis['x'].values
    ys_ap = df_apophis['y'].values
    ax.plot(xs_ap, ys_ap, color='#FF6600', lw=1.2, alpha=0.5, ls='-', zorder=3,
            label='Trayectoria natural Apophis')

    # Arco Lambert aproximado (línea directa t_DART → encuentro)
    ax.annotate('', xy=r_tierra_plot, xytext=r_ap,
                arrowprops=dict(arrowstyle='->', color='#AAFFAA', lw=2.0,
                                connectionstyle='arc3,rad=0.3'))
    mid_x = (r_ap[0] + r_tierra_plot[0]) / 2 - 0.2
    mid_y = (r_ap[1] + r_tierra_plot[1]) / 2 + 0.3
    ax.text(mid_x, mid_y, 'Fase 1:\nLambert ($\\tau$)', color='#AAFFAA',
            fontsize=8.5, ha='center',
            bbox=dict(boxstyle='round,pad=0.3', fc='#111', ec='#AAFFAA', alpha=0.8))

# ── Etiqueta Fase 0 ──
ax.annotate('', xy=(1.0+0.20, 0.0), xytext=(1.0+0.04, 0.0),
            arrowprops=dict(arrowstyle='->', color='#FFAAFF', lw=1.8))
ax.text(1.0+0.28, 0.08, 'Fase 0:\nEscape LEO', color='#FFAAFF',
        fontsize=8.5, ha='center',
        bbox=dict(boxstyle='round,pad=0.3', fc='#111', ec='#FFAAFF', alpha=0.8))

# ── Etiqueta Fase 2 ──
ax.text(r_tierra_plot[0]-0.35, r_tierra_plot[1]-0.30,
        'Fase 2:\nHipérbola geocéntrica', color='#FFFF88',
        fontsize=8.5, ha='center',
        bbox=dict(boxstyle='round,pad=0.3', fc='#111', ec='#FFFF88', alpha=0.8))

ax.set_xlabel('x [AU]', color='white', fontsize=11)
ax.set_ylabel('y [AU]', color='white', fontsize=11)
ax.set_title('Misión DART Inverso — Diagrama de Parches Cónicos', color='white', fontsize=13)
ax.tick_params(colors='white')
for spine in ax.spines.values():
    spine.set_edgecolor('white')
ax.legend(fontsize=9, facecolor='#222', edgecolor='white', labelcolor='white')
plt.tight_layout()
plt.show()

## 8. Resumen de Resultados

### 8.1 Tabla de fases

La siguiente tabla consolida los resultados numéricos de las tres fases:

| Fase | Descripción | Resultado principal |
|------|-------------|--------------------|
| **0** | Escape geocéntrico desde LEO 400 km | $\Delta V_{esc} = v_{esc} - v_{circ}$ km/s |
| **1** | Intercepción heliocéntrica (Lambert) | $\Delta V(\tau)$ varía de ~km/s a ~m/s según antelación |
| **2** | Encuentro hiperbólico geocéntrico | $b_{max}$ define la sección eficaz de impacto |

### 8.2 Limitaciones del modelo

El modelo de parches cónicos tiene las siguientes simplificaciones:

1. **Sin perturbaciones**: Se ignoran los efectos de la Luna, Júpiter, y presión de radiación solar
2. **Impulso instantáneo**: El $\Delta V$ en Apophis se modela como una maniobra impulsiva puntual
3. **Efemérides de punto de masa**: Apophis y la Tierra se tratan como puntos sin dimensión (en la Fase 1)
4. **Un solo arco Lambert**: Se usa una única solución de Lambert (órbita progresiva)

Para mayor precisión se requeriría integración numérica completa con perturbaciones (ver notebooks siguientes).

In [ ]:
print("=" * 60)
print("RESUMEN — Misión DART Inverso: Parches Cónicos")
print("=" * 60)
print(f"\n{'FASE 0 — Escape geocéntrico':}")
print(f"  v_circ LEO (400 km):  {v_circ_LEO/1e3:.3f} km/s")
print(f"  v_esc  LEO (400 km):  {v_esc_LEO/1e3:.3f} km/s")
print(f"  ΔV mínimo de escape:  {dv_escape/1e3:.3f} km/s")
print(f"\n{'FASE 1 — Intercepción heliocéntrica':}")
print(f"  Encuentro cercano:    {t_encuentro}")
for _, row in df_res.iterrows():
    print(f"  τ={row['tau_dias']:4.0f} d → ΔV = {row['dv_km_s']:.4f} km/s")
print(f"\n{'FASE 2 — Encuentro hiperbólico (caso τ=' + str(int(tau_d_best)) + ' d)':}")
print(f"  v_∞ geocéntrica:      {v_inf_best_kms:.3f} km/s")
print(f"  Semieje hipérbola:    {a_h/1e3:.0f} km")
if len(b_max_idx) > 0:
    print(f"  b_max para impacto:   {b_max:.0f} km  (q ≤ R_⊕)")
    print(f"  v_impacto (b=0):      {v_impact_ms/1e3:.3f} km/s")
print("=" * 60)